In [1]:
import re
import pandas as pd
from rapidfuzz import fuzz

In [2]:
df = pd.read_csv('../data/steam-200k.csv', header=None)
df.columns = ['user-id', 'game-title', 'behavior', 'value', 'extra']
print(f"Initial shape: {df.shape}")
df.head()

Initial shape: (200000, 5)


,user-id,game-title,behavior,value,extra
0,151603712,The Elder Scrolls V Skyrim,purchase,1.0,0
1,151603712,The Elder Scrolls V Skyrim,play,273.0,0
2,151603712,Fallout 4,purchase,1.0,0
3,151603712,Fallout 4,play,87.0,0
4,151603712,Spore,purchase,1.0,0


In [3]:
print("Unique values in 'extra':", df['extra'].unique())
df.drop(columns=['extra'], inplace=True)
print(f"Shape after dropping 'extra': {df.shape}")

Unique values in 'extra': [0]
Shape after dropping 'extra': (200000, 4)


In [4]:
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")

df.drop_duplicates(inplace=True)
print(f"Shape after dropping exact duplicates: {df.shape}")

Exact duplicate rows: 707
Shape after dropping exact duplicates: (199293, 4)


In [5]:
plays = df[df['behavior'] == 'play']
purchases = df[df['behavior'] == 'purchase']

plays_cleaned = plays.groupby(['user-id', 'game-title'], as_index=False)['value'].sum()

final_df = pd.concat([purchases, plays_cleaned], ignore_index=True)

final_df.sort_values(by=['user-id', 'game-title'], inplace=True)
final_df.reset_index(drop=True, inplace=True)

print(f"Shape after aggregating play actions: {final_df.shape}")

Shape after aggregating play actions: (199281, 4)


In [6]:
digit_word_map = {
    "one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
    "i": "1", "ii": "2", "iii": "3", "iv": "4", "v": "5", "vi": "6", "vii": "7", "viii": "8", "ix": "9", "x": "10"
}
word_digit_map = {v: k for k, v in digit_word_map.items()}

conflict_keywords = [
    "dlc", "bundle", "mac", "linux", "asia", "japan", "eu", "region", "episode", "season", 
    "expansion", "pack", "remastered", "remake", "demo"
]

In [7]:
def normalize_title(title: str) -> str:
    title = title.lower()
    for word, digit in digit_word_map.items():
        title = re.sub(rf"\b{re.escape(word)}\b", digit, title)
    return title

In [9]:
from typing import List, Tuple

def extract_numbers_with_tokens(title: str) -> List[Tuple[str, int]]:
    title = normalize_title(title)
    tokens = title.split()
    result = []
    for i, token in enumerate(tokens):
        if token.isdigit():
            prev = tokens[i-1] if i > 0 else ""
            if prev in conflict_keywords:
                result.append((prev, int(token)))
            else:
                result.append(("", int(token)))
    return result

In [ ]:
def safe_to_merge(title1: str, title2: str) -> bool:
    t1, t2 = normalize_title(title1), normalize_title(title2)

    nums1 = extract_numbers_with_tokens(t1)
    nums2 = extract_numbers_with_tokens(t2)

    # Block if same keyword
    for (k1, n1) in nums1:
        for (k2, n2) in nums2:
            if k1 == k2 and n1 != n2 and k1 != "":
                return False

    # Extract standalone numbers 
    plain_nums1 = {n for (k, n) in nums1 if k == ""}
    plain_nums2 = {n for (k, n) in nums2 if k == ""}

    #  Block if both titles have standalone numbers and they're different
    if plain_nums1 and plain_nums2 and plain_nums1 != plain_nums2:
        return False

    #  Block if only one title has a number eg- "Fallout 4" vs "Fallout New Vegas"
    if (plain_nums1 and not plain_nums2) or (plain_nums2 and not plain_nums1):
        return False

    # Block keyword conflicts
    for word in conflict_keywords:
        if (word in t1 and word not in t2) or (word in t2 and word not in t1):
            return False

    similarity = fuzz.token_set_ratio(t1, t2)
    return similarity >= 85


In [26]:
print(safe_to_merge("Rising Dead 3 DLC 4", "Rising Dead 3 DLC 5")) 
print(safe_to_merge("Call of Duty IV", "Call of Duty 4"))             
print(safe_to_merge("GTA V", "GTA Five"))                            
print(safe_to_merge("GTA V MAC", "GTA V"))                            
print(safe_to_merge("Game Pack Season 2", "Game Pack Season 3"))      


False
True
True
False
False


In [ ]:
from itertools import combinations

# Get top 10 unique titles
top_titles = df['game-title'].dropna().unique()[:10]

merges = []
skips = []

# Check all unique pairs from top 10
for t1, t2 in combinations(top_titles, 2):
    decision = safe_to_merge(t1, t2)
    if decision and len(merges) < 5:
        merges.append((t1, t2))
    elif not decision and len(skips) < 5:
        skips.append((t1, t2))
    
    if len(merges) >= 5 and len(skips) >= 5:
        break

print("✅ Sample Merges:")
for t1, t2 in merges:
    print(f"  - '{t1}'  ↔  '{t2}'")

print("\n❌ Sample Skips:")
for t1, t2 in skips:
    print(f"  - '{t1}'  ≠  '{t2}'")


✅ Sample Merges:

❌ Sample Skips:
  - 'The Elder Scrolls V Skyrim'  ≠  'Fallout 4'
  - 'The Elder Scrolls V Skyrim'  ≠  'Spore'
  - 'The Elder Scrolls V Skyrim'  ≠  'Fallout New Vegas'
  - 'The Elder Scrolls V Skyrim'  ≠  'Left 4 Dead 2'
  - 'The Elder Scrolls V Skyrim'  ≠  'HuniePop'


In [ ]:
import pandas as pd
from itertools import combinations

# Pick 10 random unique titles
sample_titles = df['game-title'].dropna().unique()
sample_titles = pd.Series(sample_titles).sample(10, random_state=42).tolist()

merge_suggestions = []

for t1, t2 in combinations(sample_titles, 2):
    merge_suggestions.append({
        "Title A": t1,
        "Title B": t2,
        "Should Merge?": safe_to_merge(t1, t2)
    })

suggestion_df = pd.DataFrame(merge_suggestions)
suggestion_df.head(15)  # display top 15 rows


,Title A,Title B,Should Merge?
0,Call of Duty IV,Call of Duty 4,True
1,Call of Duty IV,GTA Five,False
2,Call of Duty IV,GTA V,False
3,Call of Duty IV,Tomb Raider 1,False
4,Call of Duty IV,Tomb Raider One,False
5,Call of Duty IV,FIFA 2022,False
6,Call of Duty IV,FIFA Twenty Twenty Two,False
7,Call of Duty IV,Hitman III,False
8,Call of Duty IV,Hitman 3,False
9,Call of Duty 4,GTA Five,False


In [ ]:
import pandas as pd
from itertools import combinations
import random

# Sample 10 random titles 
sample_titles = [
    "Call of Duty IV", "Call of Duty 4",
    "GTA Five", "GTA V",
    "Tomb Raider 1", "Tomb Raider One",
    "FIFA 2022", "FIFA Twenty Twenty Two",
    "Hitman III", "Hitman 3"
]

suggestions = []
for t1, t2 in combinations(sample_titles, 2):
    decision = safe_to_merge(t1, t2)
    suggestions.append({
        "Title A": t1,
        "Title B": t2,
        "Should Merge": decision
    })

suggestion_df = pd.DataFrame(suggestions)
suggestion_df.query("`Should Merge` == True").head(10)

,Title A,Title B,Should Merge
0,Call of Duty IV,Call of Duty 4,True
17,GTA Five,GTA V,True
30,Tomb Raider 1,Tomb Raider One,True
44,Hitman III,Hitman 3,True


In [ ]:
test_pairs = [
    ("Assassin's Creed II Deluxe Edition", "Assassins Creed 2 Deluxe Edition"),  
    ("GTA V Bundle", "GTA V"),                                               
    ("Left 4 Dead 2", "Left 4 Dead"),                                         
    ("Resident Evil VII", "Resident Evil 7"),                                  
    ("Hitman III Linux", "Hitman 3"),                                            
    ("Witcher 3 Wild Hunt Expansion", "Witcher III Wild Hunt"),                 
    ("Batman Arkham Collection", "Batman Arkham Asylum"),                      
    ("Tomb Raider One", "Tomb Raider GOTY"),                                     
    ("BioShock Infinite", "BioShock Infinite Complete Edition"),                
    ("Dark Souls II Scholar of the First Sin", "Dark Souls 2")                 
]


In [38]:
for t1, t2 in test_pairs:
    result = safe_to_merge(t1, t2)
    print(f"{'✅' if result else '❌'} {t1}  ↔  {t2}")


✅ Assassin's Creed II Deluxe Edition  ↔  Assassins Creed 2 Deluxe Edition
❌ GTA V Bundle  ↔  GTA V
❌ Left 4 Dead 2  ↔  Left 4 Dead
✅ Resident Evil VII  ↔  Resident Evil 7
❌ Hitman III Linux  ↔  Hitman 3
❌ Witcher 3 Wild Hunt Expansion  ↔  Witcher III Wild Hunt
❌ Batman Arkham Collection  ↔  Batman Arkham Asylum
❌ Tomb Raider One  ↔  Tomb Raider GOTY
✅ BioShock Infinite  ↔  BioShock Infinite Complete Edition
✅ Dark Souls II Scholar of the First Sin  ↔  Dark Souls 2


In [43]:
from rapidfuzz import process

titles = df['game-title'].dropna().unique().tolist()
canonical_map = {}
seen = set()

for title in titles:
    if title in seen:
        continue

    # Get top 5 closest matches using token_set_ratio
    matches = process.extract(
        query=title,
        choices=titles,
        scorer=fuzz.token_set_ratio,
        limit=10
    )

    # Group titles that are safe to merge
    group = [m[0] for m in matches if safe_to_merge(title, m[0])]
    
    # Choose canonical name
    canonical = sorted(group, key=lambda x: (len(x), x))[0]
    for t in group:
        canonical_map[t] = canonical
        seen.add(t)

print("✅ Canonical map built with fuzzy pre-filtering.")



✅ Canonical map built with fuzzy pre-filtering.


In [44]:
# STEP 4: Apply mapping to the DataFrame
df['canonical_title'] = df['game-title'].map(canonical_map)

# STEP 5: Save the cleaned file
import os
os.makedirs("results", exist_ok=True)
df.to_csv("results/cleaned_titles.csv", index=False)

print(f"✅ Cleaned dataset saved to: results/cleaned_titles.csv")
print(f"📊 Total canonical titles: {len(set(df['canonical_title'].dropna()))}")


✅ Cleaned dataset saved to: results/cleaned_titles.csv
📊 Total canonical titles: 4191
